# MINI Cells — Experiment 008: Production Optimizer Search

Deterministic exact-Q8.8 search over SPSA scale, block size, objective, microbatch aggregation, and retained-proposal semantics. The experiment first reproduces the committed 512-generation local baseline; if that check fails, the search stops.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git','rev-parse','HEAD'], check=True)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Experiment 008 uses the exact integer CPU path; CUDA is not required.')


In [ ]:
subprocess.run([sys.executable,'-m','pytest','tests/research/01-foundations/test_optimizer_search.py','-q'], cwd=ROOT, check=True)


## Full search / resume

Re-running this cell is safe: each configuration stores its Q8.8 model and generation under `results/production-optimizer-search-v1/runs/` and resumes deterministically. Do not lower the decision thresholds after observing results.


In [ ]:
PROFILE = 'full'  # use 'smoke' only to validate the harness
FINAL_GENERATIONS = 2048  # set 4096 for a longer finalist validation
OUT = ROOT / 'results' / 'production-optimizer-search-v1'
cmd = [sys.executable, 'scripts/research/run_optimizer_search.py', '--profile', PROFILE,
       '--final-generations', str(FINAL_GENERATIONS), '--output', str(OUT)]
result = subprocess.run(cmd, cwd=ROOT)
print('exit code:', result.returncode)
print('Exit 3 means the search completed without a production candidate; that is a valid experimental result.')


In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
decision_path = OUT / 'decision.json'
if not decision_path.exists():
    raise RuntimeError('decision.json is missing; inspect the previous cell output')
decision = json.loads(decision_path.read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(Markdown(f"**Next action:** `{decision['next_action']}`"))
for table in ['stage1.csv','stage2.csv','finalists.csv','solved-regression.csv']:
    path = OUT / table
    if path.exists():
        display(Markdown(f'### {table}'))
        display(pd.read_csv(path).head(20))
for name in ['loss-accuracy-frontier.png','finalist-probe-loss.png','finalist-probe-accuracy.png']:
    path = OUT / name
    if path.exists(): display(Image(filename=str(path)))


In [ ]:
# Preserve this directory with Kaggle Save Version/output, regardless of PASS or NO_PRODUCTION_CANDIDATE.
print('Result directory:', OUT)
print('Decision:', decision_path)
if (OUT / 'recommended-runtime.json').exists():
    print((OUT / 'recommended-runtime.json').read_text(encoding='utf-8'))
